## Problem Statement

 **Business Context**


The healthcare industry is rapidly evolving, and professionals face increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. Quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Objective**


As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to understand information overload, apply AI techniques to streamline decision-making, analyze its impact on diagnostics and patient outcomes, evaluate its potential to standardize care practices, and create a functional prototype demonstrating its feasibility and effectiveness.

**Questions to Answer**

1. What is the protocol for managing sepsis in a critical care unit?
2. What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
3. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
4. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
5. What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

**Data Dictionary**
The Merck Manuals are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899 when Merck & Co. was still a subsidiary of the German company Merck.

The manual is a PDF with over 4,000 pages divided into 23 sections.

## GPU Selection

The T4 GPU was selected due to availability constraints. While it is not the most
powerful option, it provides sufficient capability to run 7B parameter models like
Mistral efficiently.

Higher-end GPUs such as A100 or H100 offer significantly better performance in
terms of inference speed and handling larger context windows, but T4 remains a
practical and cost-effective choice for experimentation and evaluation.

## Install libraries using dependency resolver.

In [1]:
# Install libraries using dependency resolver.
!pip install \
    huggingface_hub \
    pandas \
    tiktoken \
    pymupdf \
    langchain \
    langchain-community \
    langchain-chroma \
    sentence-transformers \
    numpy \
    -q

## Installing llama-cpp-python
Below command installs llama-cpp-python with CUDA (GPU) acceleration enabled
It ensures that the model runs on the GPU instead of CPU, significantly improving performance.

In [2]:
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124


## Loading the Large Language Model

In this section, we download and load the **Mistral 7B Instruct** model from Hugging Face,
which will serve as the core LLM for both prompt engineering experiments and the RAG pipeline.

In [3]:
#Testing llama-cpp-python with CUDA
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
import os

In [4]:
# Model configuration
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
#Libraries for processing dataframes,text
import json, os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

### LLM Configuration for Prompt Engineering and RAG


A single Mistral 7B instance is used for both prompt engineering and RAG to avoid redundant
model loading and optimize GPU memory. Key configuration choices:

- **`n_ctx=5000`**: Large context window to accommodate retrieved chunks + user query in RAG.
- **`n_batch=512`**: Moderate batch size to balance inference speed and GPU memory on T4.
- **`n_gpu_layers=40`**: Partial GPU offloading to prevent CUDA out-of-memory errors while
  still benefiting from GPU acceleration.




In [6]:
mistral_llm = Llama(
    model_path=model_path,
    n_threads=2,
    n_batch=512,
    n_gpu_layers=40,
    n_ctx=5000
)

llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) (0000:00:04.0) - 6891 MiB free
llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: -

In [7]:
# function to generate, process, and return the response from the LLM
def generate_llama_response_for_mistral(user_prompt):

    # System message
    system_message = """
    [INST]<<SYS>> Respond to the user question based on the user prompt<</SYS>>[/INST]
    """

    # Combine user_prompt and system_message to create the prompt
    prompt = f"{user_prompt}\n{system_message}"

    # Generate a response from the LLaMA model
    response = mistral_llm(
        prompt=prompt,
        max_tokens=1024,
        temperature=0.01,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]
    return response_text

## Using LLM Model without Prompt Engineering

In this section we will evaluate responses without using any prompt engineering techiniques , we will pass the user query to the LLM and analyze the responses for all the 5 questions.

**Q1. What is the protocol for managing sepsis in a critical care unit?**

In [8]:
user_prompt = "What is the protocol for managing sepsis in a critical care unit?"
response = generate_llama_response_for_mistral(user_prompt)
print(response)

ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =     217.76 ms /    46 tokens (    4.73 ms per token,   211.24 tokens per second)
llama_perf_context_print:        eval time =   33448.38 ms /   566 runs   (   59.10 ms per token,    16.92 tokens per second)
llama_perf_context_print:       total time =   34297.69 ms /   612 tokens
llama_perf_context_print:    graphs reused =        563


1. Recognize and suspect sepsis early: Look for signs of infection, organ dysfunction, and inflammation in patients, especially those with risk factors such as advanced age, chronic illnesses, or invasive devices.
    2. Initiate resuscitation: Provide adequate fluid resuscitation to maintain adequate tissue perfusion (MAP > 65 mmHg, CVP > 8-12 cm H2O, urine output >0.5 ml/kg/h). Consider using early goal-directed therapy (EGDT) or protocols such as the Surviving Sepsis Campaign guidelines to optimize fluid resuscitation and other interventions.
    3. Administer antibiotics: Start broad-spectrum antibiotics based on suspected infection site, microbiology data, and local resistance patterns. Consider deescalating therapy once culture results are available and sensitivities are known.
    4. Monitor hemodynamic status: Continuously monitor vital signs, cardiac output, oxygenation, and perfusion parameters to assess response to treatment and adjust interventions as needed.
    5. Provide

**Q2. What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [9]:
user_prompt = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response = generate_llama_response_for_mistral(user_prompt)
print(response)

Llama.generate: 2 prefix-match hit, remaining 62 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =     113.48 ms /    62 tokens (    1.83 ms per token,   546.36 tokens per second)
llama_perf_context_print:        eval time =   21211.92 ms /   382 runs   (   55.53 ms per token,    18.01 tokens per second)
llama_perf_context_print:       total time =   21711.90 ms /   444 tokens
llama_perf_context_print:    graphs reused =        380



Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can include:

1. Sudden pain that starts around the navel and then shifts to the lower right abdomen
2. Loss of appetite
3. Nausea and vomiting
4. Fever, often over 101 degrees Fahrenheit (38.3 degrees Celsius)
5. Abdominal swelling and rigidity
6. Diarrhea or constipation
7. Inability to pass gas or have a bowel movement
8. Feeling bloated or having cramps in the abdomen
9. General discomfort, uneasiness, or malaise
1

It's important to note that appendicitis can be a medical emergency and if you suspect you have it, seek immediate medical attention as delaying treatment could lead to complications such as rupture of the appendix, which can result in peritonitis - an infection of the abdominal cavity.

Appendicitis cannot typically be treated with medicine alone due to the inflammatory nature of the c

**Q3. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [10]:
user_prompt = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response = generate_llama_response_for_mistral(user_prompt)
print(response)

Llama.generate: 4 prefix-match hit, remaining 64 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =     103.04 ms /    64 tokens (    1.61 ms per token,   621.10 tokens per second)
llama_perf_context_print:        eval time =   25604.03 ms /   473 runs   (   54.13 ms per token,    18.47 tokens per second)
llama_perf_context_print:       total time =   26201.22 ms /   537 tokens
llama_perf_context_print:    graphs reused =        470


1. Possible Causes:
   - Alopecia Areata: An autoimmune disorder that causes hair loss in round patches on the scalp, beard or other areas of the body. It often starts with one or more small circular bald spots that may overlap.
   - Trauma or Stress: Physical trauma to the scalp such as burns, cuts, surgery, or emotional stress can cause sudden patchy hair loss.
   - Nutritional Deficiencies: Lack of essential nutrients like iron, zinc, biotin, and vitamin B12 may lead to hair thinning and patchiness.
   - Hormonal Imbalance: Sudden hormonal changes due to pregnancy, menopause or thyroid issues can cause temporary hair loss in patches.
   - Medications: Certain medications like chemotherapy drugs, antidepressants, beta-blockers, and steroids may lead to patchy hair loss as a side effect.
   2. Treatments:
  - Alopecia Areata: Topical treatments such as minoxidil or corticosteroid creams can help stimulate hair growth in small patches of alopecia areata. Injections of corticosteroids m

**Q4. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [11]:
user_prompt = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response = generate_llama_response_for_mistral(user_prompt)
print(response)

Llama.generate: 2 prefix-match hit, remaining 58 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =     106.25 ms /    58 tokens (    1.83 ms per token,   545.90 tokens per second)
llama_perf_context_print:        eval time =   28814.73 ms /   516 runs   (   55.84 ms per token,    17.91 tokens per second)
llama_perf_context_print:       total time =   29512.28 ms /   574 tokens
llama_perf_context_print:    graphs reused =        513


1. Medical Evaluation and Stabilization: The first step is to ensure the person's overall health and safety by addressing any life-threatening injuries or conditions. This may involve hospitalization, surgery, or emergency medical treatments.

2. Rehabilitation Program: Depending on the severity of brain injury, a rehabilitation program may be recommended. The goal is to help the person regain lost skills and abilities, manage symptoms, and improve overall functioning. A team of healthcare professionals including physicians, therapists, nurses, social workers, and psychologists work together to create an individualized treatment plan.

   - Physical Therapy: Helps restore mobility, strength, balance, coordination, and flexibility through exercises and training programs.
   
   - Occupational Therapy: Teaches skills needed for daily living activities such as dressing, cooking, writing, or using utensils.
   
   - Speech-Language Therapy: Addresses communication difficulties including sp

**Q5.What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

In [12]:
user_prompt = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response = generate_llama_response_for_mistral(user_prompt)
print(response)

Llama.generate: 2 prefix-match hit, remaining 65 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =     118.45 ms /    65 tokens (    1.82 ms per token,   548.75 tokens per second)
llama_perf_context_print:        eval time =   23092.50 ms /   410 runs   (   56.32 ms per token,    17.75 tokens per second)
llama_perf_context_print:       total time =   23647.45 ms /   475 tokens
llama_perf_context_print:    graphs reused =        408


1. Assess the situation: If you suspect someone has fractured their leg during a hiking trip, first ensure your own safety and assess the scene for any potential hazards. Approach the injured person calmly and ask them to describe what happened and where they feel pain.
    2. Check for signs of injury or complications: Look for visible deformities, swelling, bruising, open wounds, numbness, or inability to move the leg. Also, check for any signs of shock such as pale skin, rapid heartbeat, shallow breathing, and confusion.
    3. Provide first aid: If you suspect a fracture, do not try to realign the bone yourself. Instead, immobilize the leg using a splint or a makeshift sling made from clothing or other materials. Apply gentle pressure on the injury site with a clean cloth to control bleeding if necessary.
    4. Keep the person comfortable: Encourage them to lie down and elevate the injured leg above heart level to reduce swelling and pain. Provide warmth using blankets, but avoid 

### Observations



1. The responses are generally coherent but lack clinical precision and structured medical reasoning.

2. The model tends to provide generic explanations that are understandable to a general audience but may not meet the standards required for medical professionals.

3. There is no guarantee of factual correctness, as the responses are not grounded in verified medical sources, leading to potential hallucinations.

4. The responses lack citation or evidence-based justification, which is critical in healthcare decision-making.

5. Some answers are overly verbose without prioritizing critical steps (e.g., emergency protocols).

Conclusion:
While the LLM provides fluent responses, it is not reliable for clinical decision support without grounding in trusted medical data.

## Using LLM Model with Prompt Engineering

In this section, we apply **prompt engineering** to improve the quality of LLM responses.
A role-based system prompt is introduced to give the model a clear identity, behavioral
guidelines, and constraints. The same 5 medical questions are tested to directly compare
how responses change when the model is guided by a well-designed system prompt versus
answering without any context or instructions.

In [13]:
def generate_llama_response_system_prompt(
    user_prompt,
    system_prompt,
    temperature=0.3,
    max_tokens=1024,
    top_p=0.95,
    top_k=50,
    repeat_penalty=1.2
):


    prompt = f"""[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_prompt} [/INST]"""

    response = mistral_llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        echo=False
    )

    return response["choices"][0]["text"].strip()

### Creating a Role based System Prompt for getting more accurate reponses

In [14]:
system_prompt = """
You are a knowledgeable AI assistant specializing in the medical domain.

Provide clear, concise, and accurate responses aligned with medical standards.
Only provide information you are confident about.

If you are unsure or lack sufficient information, explicitly state that you do not know.
Avoid speculation or generic responses.
"""

### Testing Responses using Role based System Prompt

**Q1. What is the protocol for managing sepsis in a critical care unit?**

In [15]:
user_prompt = "What is the protocol for managing sepsis in a critical care unit?"
response = generate_llama_response_system_prompt(user_prompt,system_prompt)
print(response)

Llama.generate: 1 prefix-match hit, remaining 104 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =     141.51 ms /   104 tokens (    1.36 ms per token,   734.94 tokens per second)
llama_perf_context_print:        eval time =   28090.48 ms /   500 runs   (   56.18 ms per token,    17.80 tokens per second)
llama_perf_context_print:       total time =   28759.01 ms /   604 tokens
llama_perf_context_print:    graphs reused =        497


The protocol for managing sepsis in a critical care unit involves the following steps, based on current medical guidelines:

1. Early recognition and suspicion of sepsis: Monitor vital signs, laboratory values, and clinical symptoms to identify sepsis early. Suspect sepsis if there is evidence of infection, along with signs of organ dysfunction such as altered mental status, respiratory distress, or low urine output.

2. Rapid administration of antibiotics: Once sepsis is suspected, start broad-spectrum intravenous antibiotics without delay. Adjust the antibiotic regimen based on culture and sensitivity results.

3. Fluid resuscitation: Aim for adequate tissue perfusion by administering crystalloid fluids to maintain a mean arterial pressure (MAP) of 65 mmHg or higher, provided there is no contraindication. Monitor fluid response closely and avoid over-resuscitation.

4. Vasopressor support: If MAP remains low despite adequate fluid resuscitation, administer vasopressors to maintain ad

**Q2. What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [16]:
user_prompt = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response = generate_llama_response_system_prompt(user_prompt,system_prompt)
print(response)

Llama.generate: 87 prefix-match hit, remaining 36 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      92.38 ms /    36 tokens (    2.57 ms per token,   389.69 tokens per second)
llama_perf_context_print:        eval time =   21606.67 ms /   385 runs   (   56.12 ms per token,    17.82 tokens per second)
llama_perf_context_print:       total time =   22101.77 ms /   421 tokens
llama_perf_context_print:    graphs reused =        383


Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum in the large intestine. The following are common symptoms of appendicitis:

1. Abdominal pain, usually starting around the navel and then shifting to the lower right side
2. Loss of appetite
3. Nausea and vomiting
4. Fever (often low-grade at first)
5. Diarrhea or constipation
6. General feeling of being unwell
7. Pain upon walking, coughing, or making other jarring movements
8. Inability to pass gas or have a bowel movement
9. Increased heart rate and respiratory rate
10. Abdominal swelling and rigidity

Appendicitis is typically treated with surgery due to the risk of rupture, which can lead to peritonitis – an infection in the abdominal cavity that could be life-threatening. The standard surgical procedure for appendicitis is called an appendectomy, where the inflamed appendix is removed.

In some cases, antibiotics may be used as a first line of treatment if 

**Q3 What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [17]:
user_prompt = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response = generate_llama_response_system_prompt(user_prompt,system_prompt)
print(response)

Llama.generate: 89 prefix-match hit, remaining 38 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      86.66 ms /    38 tokens (    2.28 ms per token,   438.50 tokens per second)
llama_perf_context_print:        eval time =   33392.26 ms /   598 runs   (   55.84 ms per token,    17.91 tokens per second)
llama_perf_context_print:       total time =   34160.95 ms /   636 tokens
llama_perf_context_print:    graphs reused =        595


There are several possible causes for sudden, patchy hair loss, also known as alopecia areata. The most common treatments involve stimulating the affected area to promote regrowth of hair. Here are some effective treatment options:

1. Corticosteroids: These medications can be applied directly to the scalp or taken orally to reduce inflammation and suppress the immune system, which is believed to play a role in alopecia areata. Topical corticosteroid creams or ointments, such as clobetasol propionate (Delonex), betamethasone dipropionate (Diprolene), or hydrocortisone butyrate (Locoid), can be used for small patches of hair loss. Injections of corticosteroids into the affected area may also be effective, especially for larger bald spots.
2. Immunomodulators: These medications work by modifying the immune system to help stop the attack on hair follicles. Minoxidil (Rogaine) is a topical medication that has been shown to promote hair regrowth in some people with alopecia areata. Another 

**Q4 What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [18]:
user_prompt = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response = generate_llama_response_system_prompt(user_prompt,system_prompt)
print(response)

Llama.generate: 87 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      79.00 ms /    32 tokens (    2.47 ms per token,   405.06 tokens per second)
llama_perf_context_print:        eval time =   23839.96 ms /   426 runs   (   55.96 ms per token,    17.87 tokens per second)
llama_perf_context_print:       total time =   24378.43 ms /   458 tokens
llama_perf_context_print:    graphs reused =        423


For a person with a brain injury, the specific treatment recommendations depend on the severity and location of the injury. Here are some common interventions:

1. Initial Care: This includes managing airway, breathing, and circulation (ABCs), controlling seizures if they occur, and preventing further harm to the brain.
2. Medications: Various medications may be used for symptomatic relief, such as pain relievers, anti-seizure drugs, or sedatives to help with agitation or sleep disturbances.
3. Rehabilitative Therapies: These include physical therapy, occupational therapy, speech and language therapy, and cognitive rehabilitation to address impairments in movement, sensation, communication, memory, attention, and problem-solving skills.
4. Surgery: In some cases, surgery may be necessary to remove blood clots or repair skull fractures.
5. Assistive Devices: Depending on the extent of brain function impairment, assistive devices like wheelchairs, communication aids, and adaptive equipme

**Q5. What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

In [19]:
user_prompt = " What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response = generate_llama_response_system_prompt(user_prompt,system_prompt)
print(response)

Llama.generate: 86 prefix-match hit, remaining 40 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      90.45 ms /    40 tokens (    2.26 ms per token,   442.21 tokens per second)
llama_perf_context_print:        eval time =   27492.19 ms /   491 runs   (   55.99 ms per token,    17.86 tokens per second)
llama_perf_context_print:       total time =   28100.77 ms /   531 tokens
llama_perf_context_print:    graphs reused =        488


A leg fracture, which can occur during hiking due to falls or other traumatic events, requires prompt medical attention. Here are the necessary precautions and treatment steps:

1. Assess the severity of the injury: If you suspect a leg fracture, do not move the person unless it is safe for both parties. Check for signs of open wounds, severe swelling, or deformities in the affected limb. Call emergency services if necessary.
2. Immobilize the injured extremity: Use a splint, sling, or other immobilizing device to prevent any further movement and reduce pain. Be careful not to apply too much pressure on the injury site as it may cause discomfort or worsen swelling.
3. Control bleeding: Apply direct pressure with a clean cloth if there is active bleeding from an open wound. Elevate the injured limb above heart level, if possible, to help reduce blood flow and minimize swelling.
4. Pain management: Provide pain relief using over-the-counter medications like acetaminophen or ibuprofen as 

### Comparison of LLM Responses: Without vs With Prompt Engineering

Prompt engineering improved response structure and clinical tone, but the core problem
remained — the model still fabricated specific details like drug dosages, threshold values,
and treatment protocols with confidence, despite having no access to any verified medical
source. For example, in the sepsis response, values like "MAP >65 mmHg" and "CVP 8-12 cm H2O"
were generated from training memory, not from the Merck Manual.

This makes prompt engineering alone insufficient for medical applications. A wrong dosage or
missed contraindication presented confidently could be dangerous in a real clinical setting.
This is exactly the gap that Retrieval-Augmented Generation (RAG) is designed to fill —
grounding every response in verified, traceable medical knowledge.

## LLM Parameter Tuning

To further improve response quality, the next step is to experiment with different
LLM parameters such as temperature, top_k, top_p, and max_tokens.

These parameters influence:
- **Temperature** → Controls randomness vs determinism  
- **Top_k** → Limits token selection to top probable candidates  
- **Top_p (nucleus sampling)** → Controls diversity of generated text  
- **Max_tokens** → Controls response length  

By tuning these parameters, we aim to:
- Improve response consistency  
- Reduce hallucinations  
- Enhance clinical accuracy and relevance  

In the following section, multiple parameter combinations will be tested to analyze their impact on the quality of generated responses.

In [20]:
#define List for storing all the questions
questions=[
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

]

**Experiment 1: Baseline**

In [21]:
print("===== Experiment 1: Baseline =====\n")

for i, question in enumerate(questions, 1):
    print(f"Q{i}: {question}\n")

    response = generate_llama_response_system_prompt(
        user_prompt=question,
        system_prompt=system_prompt,
        temperature=0.3,
        top_p=0.95,
        top_k=50
    )

    print(f"Answer:\n{response}\n")
    print("-"*60)

Llama.generate: 86 prefix-match hit, remaining 19 prompt tokens to eval


===== Experiment 1: Baseline =====

Q1: What is the protocol for managing sepsis in a critical care unit?



llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      66.20 ms /    19 tokens (    3.48 ms per token,   287.02 tokens per second)
llama_perf_context_print:        eval time =   26038.76 ms /   467 runs   (   55.76 ms per token,    17.93 tokens per second)
llama_perf_context_print:       total time =   26569.70 ms /   486 tokens
llama_perf_context_print:    graphs reused =        464
Llama.generate: 87 prefix-match hit, remaining 36 prompt tokens to eval


Answer:
The protocol for managing sepsis in a critical care unit involves the following steps, based on current medical guidelines from organizations such as the Surviving Sepsis Campaign:

1. Early recognition and diagnosis: Identify sepsis suspects using clinical criteria (e.g., Sequential Organ Failure Assessment [SOFA] score ≥2 or quick Sequential [sepsis-related] Organ Failure Assessment [qSOFA] score ≥1) and laboratory results (e.g., lactate level >2 mmol/L).

2. Immediate initiation of sepsis resuscitation: Administer oxygen, intravenous fluids, and vasopressors as needed to maintain adequate tissue perfusion. Target a mean arterial pressure ≥65 mmHg or a systolic blood pressure >100 mmHg in the absence of significant cardiac disease.

3. Antibiotic administration: Start broad-spectrum antibiotics within one hour of recognition and suspicion of sepsis, based on culture results if available or local guidelines for empiric therapy.

4. Source control: Identify and address the sour

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      94.19 ms /    36 tokens (    2.62 ms per token,   382.20 tokens per second)
llama_perf_context_print:        eval time =   18969.03 ms /   341 runs   (   55.63 ms per token,    17.98 tokens per second)
llama_perf_context_print:       total time =   19392.22 ms /   377 tokens
llama_perf_context_print:    graphs reused =        339
Llama.generate: 89 prefix-match hit, remaining 38 prompt tokens to eval


...

Answer:
Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp or other parts of the body. The exact cause is not known, but it's believed to be related to a problem with the immune system.

Effective treatments for addressing sudden patchy hair loss include:

1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune response. They can be applied topically or taken orally, depending on the severity of the condition.
2. Immunotherapy: This involves using medications to stimulate an immune response in order to promote hair regrowth. One such medication is minoxidil.
3. Hair transplantation: In severe cases where there is extensive hair loss, a hair transplant may be considered as a last resort.
4. Diet and lifestyle modifications: Eating a healthy diet rich in vitamins and minerals, reducing stress levels, and avoiding tight hairstyles that pull on the sca

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      83.75 ms /    32 tokens (    2.62 ms per token,   382.11 tokens per second)
llama_perf_context_print:        eval time =   17467.02 ms /   313 runs   (   55.81 ms per token,    17.92 tokens per second)
llama_perf_context_print:       total time =   17861.44 ms /   345 tokens
llama_perf_context_print:    graphs reused =        311
Llama.generate: 87 prefix-match hit, remaining 39 prompt tokens to eval


Answer:
The treatment for a brain injury depends on the severity and location of the injury. Here are some common interventions:

1. Emergency care: For severe injuries, immediate medical attention is required to prevent further damage or complications. This may include surgery to remove hematomas (clots) or decompressing skull fractures.
2. Medications: Various medications can be used to manage symptoms and improve brain function. These might include anti-inflammatory drugs, anticonvulsants for seizure control, or stimulant medication for attention deficits.
3. Rehabilitation therapy: Physical, occupational, speech, and cognitive rehabilitation therapies can help individuals regain lost skills and improve overall functioning. These therapies may be provided in a hospital setting or through outpatient clinics.
4. Assistive devices: Depending on the extent of impairment, assistive devices such as wheelchairs, communication aids, or adaptive equipment for daily living activities can help

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      87.97 ms /    39 tokens (    2.26 ms per token,   443.32 tokens per second)
llama_perf_context_print:        eval time =   28110.09 ms /   506 runs   (   55.55 ms per token,    18.00 tokens per second)
llama_perf_context_print:       total time =   28777.35 ms /   545 tokens
llama_perf_context_print:    graphs reused =        503


Answer:
A fractured leg, also known as a broken leg, requires prompt medical attention. Here are the necessary precautions and treatment steps:

1. Immediate Care: Apply a splint or immobilize the affected limb to prevent further damage or movement. Ensure that the person is comfortable and keeps warm. Do not attempt to realign the bone as it may cause more harm.
2. Transportation: Arrange for transportation to the nearest medical facility, preferably an emergency room or urgent care center. Inform the healthcare providers about the injury in advance so they can prepare accordingly.
3. Diagnosis and Imaging: The doctor will perform a physical examination and may order X-rays or other imaging tests to confirm the fracture's location, severity, and alignment.
4. Treatment Options: Depending on the type and extent of the fracture, treatment options include immobilization with a cast or brace, surgery, or traction. The healthcare provider will discuss these options based on individual circ

**Experiment 2: Deterministic**

In [22]:
print("===== Experiment 2: Deterministic =====\n")

for i, question in enumerate(questions, 1):
    print(f"Q{i}: {question}\n")

    response = generate_llama_response_system_prompt(
        user_prompt=question,
        system_prompt=system_prompt,
        temperature=0.1,
        top_p=0.9,
        top_k=40
    )

    print(f"Answer:\n{response}\n")
    print("-"*60)

Llama.generate: 87 prefix-match hit, remaining 18 prompt tokens to eval


===== Experiment 2: Deterministic =====

Q1: What is the protocol for managing sepsis in a critical care unit?



llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      72.73 ms /    18 tokens (    4.04 ms per token,   247.48 tokens per second)
llama_perf_context_print:        eval time =   23969.81 ms /   432 runs   (   55.49 ms per token,    18.02 tokens per second)
llama_perf_context_print:       total time =   24477.11 ms /   450 tokens
llama_perf_context_print:    graphs reused =        429
Llama.generate: 87 prefix-match hit, remaining 36 prompt tokens to eval


Answer:
The protocol for managing sepsis in a critical care unit involves the following steps, based on current medical guidelines:

1. Early recognition and suspicion of sepsis: Monitor patients closely for signs and symptoms of infection, such as fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.
2. Rapid assessment and initiation of treatment: If sepsis is suspected, obtain blood cultures before starting antibiotics if possible. Begin broad-spectrum antibiotic therapy as soon as possible based on the patient's clinical presentation and culture results. Provide oxygen support to maintain adequate oxygenation.
3. Fluid resuscitation: Administer intravenous fluids to maintain adequate circulating volume, aiming for a mean arterial pressure (MAP) of 65 mmHg or higher in adults. Use crystalloids initially and consider colloids if fluid requirements are high.
4. Vasopressor support: If the patient's MAP remains below target 

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      93.49 ms /    36 tokens (    2.60 ms per token,   385.08 tokens per second)
llama_perf_context_print:        eval time =   15638.04 ms /   282 runs   (   55.45 ms per token,    18.03 tokens per second)
llama_perf_context_print:       total time =   15988.42 ms /   318 tokens
llama_perf_context_print:    graphs reused =        280
Llama.generate: 89 prefix-match hit, remaining 38 prompt tokens to eval


...

Answer:
There are several possible causes for sudden, patchy hair loss, also known as alopecia areata. The most common treatment involves the use of topical or injected corticosteroids to reduce inflammation and promote regrowth. Other treatments include:

1. Minoxidil (Rogaine): This medication can help slow down hair loss and promote new growth in some people with alopecia areata. It is applied directly to the scalp twice a day.
2. DHT blockers: Finasteride (Proscar, Propecia) or dutasteride (Avodart) may be prescribed for those with androgenetic alopecia, which can cause patchy hair loss in some cases. These medications work by blocking the production of dihydrotestosterone (DHT), a hormone that contributes to hair loss.
3. Immunotherapy: Injections of certain substances like squamous cell carcinoma vaccine or diphenylcyclopropenone can help stimulate regrowth in some people with alopecia areata by altering the immune response.
4. Wigs, hairpieces, and microscopic hair transplants: 

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      82.53 ms /    32 tokens (    2.58 ms per token,   387.74 tokens per second)
llama_perf_context_print:        eval time =   19023.15 ms /   342 runs   (   55.62 ms per token,    17.98 tokens per second)
llama_perf_context_print:       total time =   19427.63 ms /   374 tokens
llama_perf_context_print:    graphs reused =        340
Llama.generate: 87 prefix-match hit, remaining 39 prompt tokens to eval


Answer:
The treatment for a brain injury depends on the severity and location of the injury. Here are some common treatments:

1. Emergency care: For severe injuries, immediate medical attention is required to prevent further damage or complications. This may include surgery to remove hematomas (clots) or decompressing skull fractures.
2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, pain, or infections.
3. Rehabilitation therapy: Physical, occupational, and speech therapies can help improve function and mobility after a brain injury. These therapies aim to restore lost skills and teach new ones.
4. Assistive devices: Devices like wheelchairs, walkers, braces, or communication aids may be necessary for individuals with permanent impairments.
5. Lifestyle modifications: Making changes in daily routines, such as adjusting the home environment to accommodate disabilities and modifying work schedules, can help improve quality o

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      88.58 ms /    39 tokens (    2.27 ms per token,   440.28 tokens per second)
llama_perf_context_print:        eval time =   20180.89 ms /   364 runs   (   55.44 ms per token,    18.04 tokens per second)
llama_perf_context_print:       total time =   20630.55 ms /   403 tokens
llama_perf_context_print:    graphs reused =        362


Answer:
A fractured leg, also known as a broken leg, requires prompt medical attention. Here are the necessary precautions and treatment steps:

1. Immediate Care: If you suspect a fracture, do not move the person excessively or apply weight on the affected leg. Immobilize the area using a splint or sling to prevent further injury. Apply a cool compress to reduce swelling and pain. Seek emergency medical help as soon as possible.

2. Diagnosis: A healthcare professional will diagnose the fracture through physical examination, X-rays, or other imaging tests. They may also assess for any associated injuries such as nerve damage or soft tissue injury.

3. Treatment: Depending on the severity and location of the fracture, treatment options include immobilization using a cast or brace, surgery to realign the bone, or traction to gradually align the bones. Pain management through medication may also be necessary.

4. Recovery: The recovery process for a broken leg can take several weeks to m

**Experiment 3: Balanced**

In [23]:
print("===== Experiment 3: Balanced =====\n")

for i, question in enumerate(questions, 1):
    print(f"Q{i}: {question}\n")

    response = generate_llama_response_system_prompt(
        user_prompt=question,
        system_prompt=system_prompt,
        temperature=0.4,
        top_p=0.9,
        top_k=50
    )

    print(f"Answer:\n{response}\n")
    print("-"*60)

Llama.generate: 87 prefix-match hit, remaining 18 prompt tokens to eval


===== Experiment 3: Balanced =====

Q1: What is the protocol for managing sepsis in a critical care unit?



llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      71.89 ms /    18 tokens (    3.99 ms per token,   250.40 tokens per second)
llama_perf_context_print:        eval time =   22528.88 ms /   405 runs   (   55.63 ms per token,    17.98 tokens per second)
llama_perf_context_print:       total time =   22993.48 ms /   423 tokens
llama_perf_context_print:    graphs reused =        403
Llama.generate: 87 prefix-match hit, remaining 36 prompt tokens to eval


Answer:
The protocol for managing sepsis in a critical care unit involves the following steps, based on current medical guidelines:

1. Early recognition and diagnosis: Identify sepsis suspects using clinical criteria such as suspected infection and organ dysfunction (SOFA score ≥2).
2. Rapid administration of antibiotics: Start broad-spectrum antimicrobial therapy within one hour of recognition for confirmed or suspected sepsis.
3. Fluid resuscitation: Aggressively administer fluid to maintain adequate tissue perfusion, aiming for a mean arterial pressure (MAP) ≥65 mmHg and central venous oxygen saturation (ScvO2) >70%.
4. Adjustment of vasopressors: Use vasopressors if necessary to maintain MAP above the target value.
5. Corticosteroids: Consider administering corticosteroids in cases of septic shock that do not respond to fluid resuscitation and vasopressor therapy within 1 hour.
6. Inotropes: Use inotropic agents if required for cardiac support.
7. Glucose control: Maintain strict 

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      92.36 ms /    36 tokens (    2.57 ms per token,   389.80 tokens per second)
llama_perf_context_print:        eval time =   14811.64 ms /   267 runs   (   55.47 ms per token,    18.03 tokens per second)
llama_perf_context_print:       total time =   15141.39 ms /   303 tokens
llama_perf_context_print:    graphs reused =        265
Llama.generate: 89 prefix-match hit, remaining 38 prompt tokens to eval


...

Answer:
Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that causes hair loss in small patches on the scalp or other parts of the body. The exact cause of this condition is not fully understood, but it's believed to be related to a problem with the immune system attacking the hair follicles.

There are several treatments for alopecia areata:

1. Corticosteroids: These medications can help suppress the immune response and promote regrowth of hair. They can be applied topically or taken orally.
2. Immunotherapy: This involves using substances that stimulate an immune response to promote hair growth. One such substance is squaric acid dibutyl ester (SADBE).
3. Minoxidil: This medication, which is available over the counter, can help slow down hair loss and promote regrowth in some people with alopecia areata.
4. Hair transplantation: In severe cases of alopecia areata where large areas of the scalp have been affected, hair transplantation may be an option

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      87.99 ms /    32 tokens (    2.75 ms per token,   363.67 tokens per second)
llama_perf_context_print:        eval time =   27951.68 ms /   504 runs   (   55.46 ms per token,    18.03 tokens per second)
llama_perf_context_print:       total time =   28573.25 ms /   536 tokens
llama_perf_context_print:    graphs reused =        501
Llama.generate: 87 prefix-match hit, remaining 39 prompt tokens to eval


Answer:
The treatment for a brain injury depends on the severity and location of the injury. Here are some common interventions:

1. Emergency care: For severe injuries, immediate medical attention is necessary to prevent further damage or complications. This may include surgery to remove hematomas (clots) or decompressing skull fractures.
2. Rehabilitation: Physical therapy, occupational therapy, speech-language therapy, and cognitive rehabilitation are often recommended for individuals with brain injuries to help them regain lost skills and improve overall functioning.
3. Medications: Depending on the specific symptoms of a brain injury, various medications may be prescribed, such as anti-seizure drugs, pain relievers, or psychotropic agents to manage conditions like depression, anxiety, or sleep disturbances.
4. Assistive devices: Devices like wheelchairs, walkers, braces, and communication aids can help individuals with brain injuries overcome physical limitations and improve their

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      93.82 ms /    39 tokens (    2.41 ms per token,   415.69 tokens per second)
llama_perf_context_print:        eval time =   32155.30 ms /   582 runs   (   55.25 ms per token,    18.10 tokens per second)
llama_perf_context_print:       total time =   32921.63 ms /   621 tokens
llama_perf_context_print:    graphs reused =        579


Answer:
A fractured leg, also known as a broken leg, requires prompt medical attention. Here are the necessary precautions and treatment steps:

1. Immediate Care: Apply a splint or immobilize the affected limb with a sling or brace to prevent further damage and provide support. Try to keep the leg elevated above heart level to reduce swelling and pain. Do not attempt to realign the bone yourself as it may cause more harm than good.
2. Seek Medical Help: Arrange for transportation to the nearest medical facility, ideally an emergency room or urgent care center. Inform them about your condition so they can prepare accordingly.
3. Diagnosis and Imaging: A healthcare professional will assess the severity of the fracture by taking a detailed medical history and performing a physical examination. X-rays are usually taken to confirm the diagnosis and determine the location, type, and extent of the fracture.
4. Treatment Options: Depending on the nature and complexity of the fracture, treatme

**Experiment 4: Creative**

In [24]:
print("===== Experiment 4: Creative =====\n")

for i, question in enumerate(questions, 1):
    print(f"Q{i}: {question}\n")

    response = generate_llama_response_system_prompt(
        user_prompt=question,
        system_prompt=system_prompt,
        temperature=0.7,
        top_p=0.95,
        top_k=60
    )

    print(f"Answer:\n{response}\n")
    print("-"*60)

Llama.generate: 87 prefix-match hit, remaining 18 prompt tokens to eval


===== Experiment 4: Creative =====

Q1: What is the protocol for managing sepsis in a critical care unit?



llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      75.56 ms /    18 tokens (    4.20 ms per token,   238.21 tokens per second)
llama_perf_context_print:        eval time =   39728.07 ms /   720 runs   (   55.18 ms per token,    18.12 tokens per second)
llama_perf_context_print:       total time =   40693.77 ms /   738 tokens
llama_perf_context_print:    graphs reused =        716
Llama.generate: 87 prefix-match hit, remaining 36 prompt tokens to eval


Answer:
The protocol for managing sepsis in a critical care unit involves early recognition, swift initiation of appropriate antibiotic therapy, and supportive measures to maintain organ function. Here's an outline:

1. Recognition: Look out for signs of infection (fever, chills, leukocytosis or leukopenia) along with systemic inflammatory response syndrome (SIRS), which includes:
   - Temperature >38°C or <36°C
   - Heart rate >90 beats per minute
   - Respiratory rate >20 breaths/minute or arterial carbon dioxide tension (PaCO2) <32 mmHg
   - Mean arterial pressure <70 mm Hg, or a systolic blood pressure of <100 mm Hg

2. Early goal-directed therapy: This includes:
   - Maintain adequate oxygenation and ventilation (SpO2 >96% on room air)
   - Aim for central venous oxygen saturation (ScvO2) >70% or a mixed venous oxygen saturation (SvO2) >65%
   - Administer fluids to maintain adequate intravascular volume: Crystalloids and colloids can be used in an appropriate dose, based on the p

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      92.24 ms /    36 tokens (    2.56 ms per token,   390.28 tokens per second)
llama_perf_context_print:        eval time =   24691.60 ms /   446 runs   (   55.36 ms per token,    18.06 tokens per second)
llama_perf_context_print:       total time =   25253.20 ms /   482 tokens
llama_perf_context_print:    graphs reused =        443
Llama.generate: 89 prefix-match hit, remaining 38 prompt tokens to eval


...

Answer:
Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp or other parts of the body. The exact cause is not known, but it's believed to be related to a problem with the immune system attacking hair follicles.

The most common treatment for alopecia areata is corticosteroids, which can help promote hair regrowth by reducing inflammation and suppressing the immune response. The treatment options include:

1. Topical steroids: Applied directly to the affected area in the form of creams or ointments.
2. Injectable steroids: Steroid injections into the bald spots.
3. Systemic steroids: Oral medication for severe cases that do not respond to topical treatments.
4. Immunomodulators: Drugs like minoxidil, anthralin, or contact sensitizers may be used in combination with corticosteroids to improve results.
5. Phototherapy: Light therapy using ultraviolet A (UVA) and psoralen or laser therapy can help stimulate

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      82.18 ms /    32 tokens (    2.57 ms per token,   389.39 tokens per second)
llama_perf_context_print:        eval time =   31320.84 ms /   567 runs   (   55.24 ms per token,    18.10 tokens per second)
llama_perf_context_print:       total time =   32070.17 ms /   599 tokens
llama_perf_context_print:    graphs reused =        564
Llama.generate: 87 prefix-match hit, remaining 39 prompt tokens to eval


Answer:
For a person with a brain injury, the specific treatment recommendations depend on various factors such as the severity and location of the injury, the age and overall health of the individual, and the extent of functional impairment. Here are some common treatments:

1. Emergency care: In case of an acute brain injury, immediate medical attention is necessary to prevent further damage or complications. This may include measures such as controlling bleeding, managing seizures, maintaining adequate oxygenation, and addressing any life-threatening conditions.

2. Rehabilitation therapies: Depending on the impairments caused by the brain injury, various rehabilitation therapies can help improve function and enhance quality of life. These may include physical therapy to address motor deficits or mobility issues; occupational therapy for fine motor skills, hand-eye coordination, and daily living activities; speech therapy for language difficulties or swallowing problems; cognitive r

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      95.54 ms /    39 tokens (    2.45 ms per token,   408.19 tokens per second)
llama_perf_context_print:        eval time =   40382.68 ms /   732 runs   (   55.17 ms per token,    18.13 tokens per second)
llama_perf_context_print:       total time =   41382.01 ms /   771 tokens
llama_perf_context_print:    graphs reused =        728


Answer:
A leg fracture, which can occur during hiking due to falls or other traumatic events, requires prompt medical attention. Here are the necessary precautions and treatment steps:

1. Immediate care: Encourage the person to remain calm and avoid moving the affected limb unnecessarily to prevent further damage. Apply a clean pad or cloth over the injured area if there is bleeding but do not apply pressure directly on the bone as it may cause more pain and discomfort. Splinting or immobilizing the leg with available materials like sticks, clothing items, or hiking poles can help minimize movement to reduce swelling and prevent complications.
2. Transportation: Arrange for transportation to a medical facility as soon as possible. If the person's condition is stable and assistance is not readily available, consider contacting emergency services or local authorities for help.
3. Medical evaluation and diagnosis: Once at the hospital, healthcare professionals will evaluate the leg fract

**Experiment 5: Constrained**

In [25]:
print("===== Experiment 5: Constrained =====\n")

for i, question in enumerate(questions, 1):
    print(f"Q{i}: {question}\n")

    response = generate_llama_response_system_prompt(
        user_prompt=question,
        system_prompt=system_prompt,
        temperature=0.2,
        top_p=0.8,
        top_k=20
    )

    print(f"Answer:\n{response}\n")
    print("-"*60)

Llama.generate: 87 prefix-match hit, remaining 18 prompt tokens to eval


===== Experiment 5: Constrained =====

Q1: What is the protocol for managing sepsis in a critical care unit?



llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      72.07 ms /    18 tokens (    4.00 ms per token,   249.76 tokens per second)
llama_perf_context_print:        eval time =   23742.15 ms /   428 runs   (   55.47 ms per token,    18.03 tokens per second)
llama_perf_context_print:       total time =   24265.69 ms /   446 tokens
llama_perf_context_print:    graphs reused =        425
Llama.generate: 87 prefix-match hit, remaining 36 prompt tokens to eval


Answer:
The protocol for managing sepsis in a critical care unit involves early recognition, quick initiation of appropriate antibiotic therapy, and supportive measures to maintain organ function. Here are the key steps:

1. Recognition: Identify patients with suspected infection and assess for signs of sepsis using the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score or Quick Sequential [Sepsis-related] Organ Failure Assessment (qSOFA) score.
2. Resuscitation: Administer oxygen, intravenous fluids, and vasopressors as needed to maintain adequate tissue perfusion and organ function.
3. Antimicrobial therapy: Start broad-spectrum antibiotics based on the suspected infection source and local microbiology data.
4. Source control: Identify and address the underlying infection source (e.g., drain abscesses, debride necrotic tissue).
5. Hemodynamic support: Monitor and manage blood pressure, cardiac output, and oxygenation as needed with vasopressors, inotropes, or mechanica

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      93.63 ms /    36 tokens (    2.60 ms per token,   384.48 tokens per second)
llama_perf_context_print:        eval time =   16420.28 ms /   296 runs   (   55.47 ms per token,    18.03 tokens per second)
llama_perf_context_print:       total time =   16789.72 ms /   332 tokens
llama_perf_context_print:    graphs reused =        294
Llama.generate: 89 prefix-match hit, remaining 38 prompt tokens to eval


...

Answer:
There are several potential causes for sudden, patchy hair loss, also known as alopecia areata. Some of the most common causes include:

1. Autoimmune conditions: Alopecia areata is an autoimmune disease where the body's immune system attacks the hair follicles, leading to hair loss. Other autoimmune diseases like lupus or thyroid disorders can also cause hair loss.
2. Stress: Physical or emotional stress can trigger alopecia areata or worsen existing cases.
3. Vitamin deficiencies: Deficiencies in iron, zinc, biotin, or vitamin D can lead to hair loss.
4. Hormonal imbalances: Thyroid hormone imbalances or androgen excess can cause patchy hair loss.
5. Medications: Certain medications like chemotherapy drugs, antidepressants, and blood thinners can cause hair loss as a side effect.
6. Trauma: Physical trauma to the scalp, such as burns or injury, can lead to patchy hair loss.
7. Infections: Fungal infections like ringworm or bacterial infections can also cause hair loss.

As fo

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      85.53 ms /    32 tokens (    2.67 ms per token,   374.12 tokens per second)
llama_perf_context_print:        eval time =   25183.51 ms /   454 runs   (   55.47 ms per token,    18.03 tokens per second)
llama_perf_context_print:       total time =   25762.47 ms /   486 tokens
llama_perf_context_print:    graphs reused =        451
Llama.generate: 87 prefix-match hit, remaining 39 prompt tokens to eval


Answer:
The treatment for a brain injury depends on the severity and location of the injury. Here are some common interventions:

1. Initial Care: This includes managing airway, breathing, and circulation (ABCs), controlling seizures if they occur, and preventing further injury to the brain.
2. Medications: Depending on the symptoms, medications may be prescribed for pain relief, reducing swelling or inflammation, controlling seizures, improving cognitive function, or managing behavioral changes.
3. Rehabilitation Therapies: These include physical therapy to help regain motor skills and mobility; occupational therapy to learn new ways of doing daily tasks; speech-language therapy for communication issues; and cognitive rehab for memory, attention, and problem-solving abilities.
4. Surgery: In some cases, surgery may be necessary to remove blood clots or repair skull fractures.
5. Alternative Therapies: These can include acupuncture, massage, biofeedback, and other complementary therapi

llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      91.19 ms /    39 tokens (    2.34 ms per token,   427.68 tokens per second)
llama_perf_context_print:        eval time =   19233.46 ms /   346 runs   (   55.59 ms per token,    17.99 tokens per second)
llama_perf_context_print:       total time =   19666.08 ms /   385 tokens
llama_perf_context_print:    graphs reused =        344


Answer:
A fractured leg, also known as a broken leg, requires prompt medical attention. Here are the necessary precautions and treatment steps:

1. Immobilize the affected limb: Use a splint or a cast to keep the bone in place and prevent any further movement that could worsen the injury. This will help reduce pain and swelling.
2. Control bleeding: Apply direct pressure on the wound with a clean cloth if there is active bleeding. Elevate the leg above heart level to minimize blood flow to the injured area.
3. Pain management: Over-the-counter pain medications like acetaminophen or ibuprofen can help manage pain. Prescription painkillers may be necessary in some cases, especially after surgery.
4. Monitor for signs of infection: Keep an eye out for symptoms such as redness, warmth, swelling, increased pain, and fever. If you notice any of these signs, contact your healthcare provider immediately.
5. Rest and avoid weight-bearing activities: Avoid putting weight on the injured leg to al

### Observations: LLM Parameter Tuning

`max_tokens=1024` was kept constant across all 5 experiments, with only `temperature`,
`top_p`, and `top_k` varied.

- **Temperature had the most visible impact.** Lower values (0.1–0.2) produced focused,
  structured responses with better clinical accuracy. Higher values (0.7) made responses
  longer and more elaborate but occasionally drifted from the core answer.

- **top_p and top_k had minimal impact** on response quality when temperature was already
  low. Their effect was more noticeable at higher temperatures.

- **Deterministic (temp=0.1)** gave the most reliable and concise responses overall —
  a clear winner for medical use cases where precision matters more than creativity.

- **Creative (temp=0.7)** responses were well-written but too verbose, and in some cases
  added steps that weren't strictly necessary.

- Across all experiments, the responses lacked **source grounding** — the model was
  drawing from training knowledge, which carries hallucination risk regardless of
  parameter settings. This reinforces the need for RAG.

**Bottom line:** For medical question answering, keep temperature low (0.1),
top_p at 0.95, and top_k at 50. Parameter tuning alone cannot fix hallucination —
that's a problem only RAG can address.

## Retrieval Augmented Generation

### Data Loading

#### Loading the Merck Manual of Diagnosis

The Merck Manual of Diagnosis & Therapy (19th Edition) is loaded from Google Drive using
`PyMuPDFLoader`, which efficiently extracts text content page by page. The document contains
**4,114 pages** covering a wide range of medical topics. A quick sanity check is performed
by printing the first 3 pages and a random page to verify that the text is being extracted
correctly before proceeding to chunking and embedding.

In [26]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
medical_pdf_path = "/content/drive/MyDrive/ColabNotebooks/RAG/medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(medical_pdf_path)



In [28]:
medical_journal=pdf_loader.load()

#### Checking the first 3 pages

In [29]:
for i in range(3):
    print(f"Page Number : {i+1}",end="\n")
    print(medical_journal[i].page_content,end="\n")

Page Number : 1
conqueror.santhosh@gmail.com
BQMJCR8G5Z
This file is meant for personal use by conqueror.santhosh@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
conqueror.santhosh@gmail.com
BQMJCR8G5Z
This file is meant for personal use by conqueror.santhosh@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...................................................................................................

In [30]:
medical_journal[100].page_content

'Vitamin D is a prohormone with several active metabolites that act as hormones. Vitamin D is metabolized\nby the liver to 25(OH)D, which is then converted by the kidneys to 1,25(OH)2D (1,25-\ndihydroxycholecalciferol, calcitriol, or active vitamin D hormone). 25(OH)D, the major circulating form, has\nsome metabolic activity, but 1,25(OH)2D is the most metabolically active. The conversion to 1,25(OH)2D\nis regulated by its own concentration, parathyroid hormone (PTH), and serum concentrations of Ca and\nphosphate.\n[\nTable 4-6. Actions of Vitamin D and its Metabolites]\nVitamin D affects many organ systems (see Table 4-6), but mainly it increases Ca and phosphate\nabsorption from the intestine and promotes normal bone formation and mineralization. Vitamin D and\nrelated analogs may be used to treat psoriasis, hypoparathyroidism, renal osteodystrophy, and possibly\nleukemia and breast, prostate, and colon cancers; they may also be used for immunosuppression.\nVitamin D Deficiency and D

#### Checking the number of pages

In [31]:
len(medical_journal)

4114

## Data Chunking

The 4,114-page Merck Manual PDF is split into smaller chunks using `RecursiveCharacterTextSplitter`
with a `chunk_size=512` tokens and `chunk_overlap=20` tokens. The `cl100k_base` tiktoken encoder
is used to ensure accurate token counting. This results in **8,487 chunks** that capture focused,
meaningful sections of the document. The small overlap ensures that context is not lost at chunk
boundaries, which is important for maintaining continuity across related medical content.

In [32]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)

In [33]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [34]:
len(document_chunks)

8487

In [35]:
document_chunks[94].page_content

'absorption. Most patients receiving phenytoin, phenobarbital, primidone, or phenothiazines develop folate\ndeficiency, probably because hepatic microsomal drug-metabolizing enzymes are affected. Folate\nsupplements may\n[Table 1-6. Effects of Some Drugs on Nutrition]\nmake phenytoin less effective. Anticonvulsants can cause vitamin D deficiency. Malabsorption of vitamin\nB12 can occur with use of aminosalicylic acid, slow-release K iodide, colchicine, trifluoperazine, ethanol,\nand oral contraceptives. Oral contraceptives with a high progestin dose can cause depression, probably\nbecause of metabolically induced tryptophan deficiency.\nFood Additives and Contaminants\nAdditives: Chemicals are often combined with foods to facilitate their processing and preservation or to\nenhance their desirability. Only amounts of additives shown to be safe by laboratory tests are permitted in\ncommercially prepared foods.\nWeighing the benefits of additives (eg, reduced waste, increased variety of a

In [36]:
document_chunks[-3000].page_content

'large or poorly differentiated or if several tumors are present. Bladder cancer tends to metastasize to the\nlymph nodes, lungs, liver, and bone. Expression of tumor gene p53 may be associated with progression.\nIn the bladder, carcinoma in situ is high grade but noninvasive and usually multifocal; it tends to recur.\nSymptoms and Signs\nMost patients present with unexplained hematuria (gross or microscopic). Some patients present with\nanemia, and hematuria is detected during evaluation. Irritative voiding symptoms (dysuria, burning,\nfrequency) and pyuria are also common at presentation. Pelvic pain occurs with advanced cancer, when\na pelvic mass may be palpable.\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nChapter 243. Genitourinary Cancer\n2618\nconqueror.santhosh@gmail.com\nBQMJCR8G5Z\nThis file is meant for personal use by conqueror.santhosh@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

## Embedding

The `thenlper/gte-large` model is used to convert document chunks and user queries into
**1024-dimensional dense vector representations**. It is a high-performing sentence transformer
model that captures semantic meaning, enabling the retriever to find relevant chunks based on
**meaning rather than exact keyword matching**.

In [37]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipykernel_86815/4198310515.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [38]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [39]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

* The embedding model provides a fixed-length vector for any number of chunks.  
* This is necessary because we want to compare them for similarity.

## Vector Database

The document chunks are stored in a **Chroma vector database** using the `thenlper/gte-large`
embedding model. Chroma persists the embeddings to disk (`medical_journal_db`) so the vector
store does not need to be rebuilt on every run, saving significant computation time.

In [40]:
out_dir = 'medical_journal_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [41]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [42]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_86815/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [43]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [44]:
vectorstore.similarity_search("What is the protocol for managing sepsis in a critical care unit?",k=15)

[Document(metadata={'format': 'PDF 1.7', 'subject': '', 'keywords': '', 'total_pages': 4114, 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationDate': 'D:20120615054440Z', 'creationdate': '2012-06-15T05:44:40+00:00', 'page': 2400, 'modDate': 'D:20260409141958Z', 'creator': 'Atop CHM to PDF Converter', 'file_path': '/content/drive/MyDrive/ColabNotebooks/RAG/medical_diagnosis_manual.pdf', 'moddate': '2026-04-09T14:19:58+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'trapped': '', 'author': '', 'source': '/content/drive/MyDrive/ColabNotebooks/RAG/medical_diagnosis_manual.pdf'}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, ped

## Retriever

The retriever is built on top of the Chroma vector store using **cosine similarity search**.
Given a user query, it fetches the top `k=2` most semantically relevant document chunks,
which are then passed as context to the LLM for generating grounded responses.

In [45]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [46]:
rel_docs = retriever.invoke("What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?")
rel_docs

[Document(metadata={'creationdate': '2012-06-15T05:44:40+00:00', 'page': 174, 'format': 'PDF 1.7', 'author': '', 'keywords': '', 'creator': 'Atop CHM to PDF Converter', 'subject': '', 'creationDate': 'D:20120615054440Z', 'trapped': '', 'modDate': 'D:20260409141958Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'total_pages': 4114, 'moddate': '2026-04-09T14:19:58+00:00', 'source': '/content/drive/MyDrive/ColabNotebooks/RAG/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'file_path': '/content/drive/MyDrive/ColabNotebooks/RAG/medical_diagnosis_manual.pdf'}, page_content="• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases,

- We can observe that the two relevant chunks contain the answer to the query.  
- If we increase the **`k`** value, there is a chance that we might find the answer in even more chunks.  
- This is a hyperparameter that we need to tune to get the best context.

## Response Function

### System Message


The system message defines the **role, behavior, and constraints** of the LLM when answering
medical queries. It acts as a set of instructions that the model must strictly follow throughout
the conversation. Key design decisions include:

- **Role Definition**: The model is assigned the role of an *expert medical AI assistant*,
  which primes it to respond in a clinically appropriate tone and style.
- **Context Grounding**: The model is explicitly instructed to base its answers **only** on
  the provided context, preventing hallucination and unsupported claims.
- **No External Knowledge**: The model is restricted from using its training knowledge,
  ensuring all responses are traceable back to the retrieved document chunks.
- **Pointwise Format**: Responses are requested in a numbered/pointwise format, improving
  readability and making it easier for healthcare professionals to quickly scan key information.
- **Graceful Fallback**: If the answer is not found in the context, the model is instructed
  to respond with *"I don't know"* rather than hallucinating an answer — a critical safeguard
  in a medical setting.
- **Source Concealment**: The model is asked not to explicitly mention the context or source
  in its response, keeping the output clean and professional.

In [47]:
qna_system_message = """
You are an expert medical AI assistant.
Answer the user's question strictly using the provided context.
Guidelines:
- Base your answer only on the given context.
- Do not add external knowledge or assumptions.
- Keep the answer precise, structured, and clinically relevant.
- Avoid mentioning the source or the context explicitly.
- If the answer is not present in the context, respond with: "I don't know."
- Answer the question in numbered format or pointwise
Focus on accuracy, clarity, and completeness.

User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.
User questions will begin with the token: ###Question.
Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.
"""

### User Message Template
The user message template structures the **input** that is passed to the model at query time.
It separates the retrieved context from the user question using clear delimiter tokens:

- **`###Context`**: Marks the beginning of the retrieved document chunks from the vector
  database. These chunks are dynamically injected at runtime based on the user's query.
- **`###Question`**: Marks the actual question asked by the user.

This clear separation helps the model distinguish between *what it knows from the document*
and *what it is being asked*, leading to more focused and accurate responses.

In [48]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### RAG Response Function

The `generate_rag_response_proper` function implements the core RAG pipeline. It retrieves
the top `k` relevant document chunks from the Chroma vector store based on the user's query,
injects them into the prompt template, and passes the final prompt to the Mistral 7B model
using the required `[INST] <<SYS>>` format for accurate instruction following.

Key parameters like `temperature=0.1` ensure deterministic and factually grounded responses,
while `repeat_penalty=1.15` and `stop` tokens prevent repetitive or incomplete outputs.
A post-processing step removes any unwanted prefixes from the model's output, and a
`try-except` block handles any runtime errors gracefully.

In [52]:
def generate_rag_response_proper(user_input, k=3, max_tokens=1024, temperature=0.1, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template

    # Retrieve relevant document chunks
    relevant_document_chunks = vectorstore.similarity_search(user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Use correct Mistral [INST] format
    prompt = f"""[INST] <<SYS>>
{qna_system_message}
<</SYS>>

{user_message} [/INST]"""

    try:
        response = mistral_llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repeat_penalty=1.15,
            stop=["[INST]", "</s>"]
        )
        result = response['choices'][0]['text'].strip()
        # Remove any "###Answer" prefix if the model outputs it
        if result.startswith("###Answer"):
            result = result[len("###Answer"):].strip()
        return result
    except Exception as e:
        return f'Sorry, I encountered the following error: \n {e}'

## Answering Question using LLM without RAG

In [49]:
response = generate_llama_response_system_prompt(
        user_prompt="Who are the Editorial and Production Staff ?",
        system_prompt=system_prompt,
        temperature=0,
        top_p=0.95,
        top_k=60
    )

print(f"Answer:\n{response}\n")
print("-"*60)

Llama.generate: 86 prefix-match hit, remaining 13 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      76.20 ms /    13 tokens (    5.86 ms per token,   170.59 tokens per second)
llama_perf_context_print:        eval time =    7303.53 ms /   142 runs   (   51.43 ms per token,    19.44 tokens per second)
llama_perf_context_print:       total time =    7490.92 ms /   155 tokens
llama_perf_context_print:    graphs reused =        141


Answer:
I'm an AI language model and don't have the ability to directly access or provide information about specific editorial and production staff in a medical context. However, in general terms, editorial staff are responsible for overseeing the content of publications such as medical journals or textbooks. They may include editors-in-chief, associate editors, section editors, and manuscript reviewers who ensure that the information is accurate, unbiased, and relevant to their field. Production staff, on the other hand, are responsible for the physical production of a publication, including design, layout, printing, and distribution. They may also be involved in digital publishing platforms and multimedia content creation.

------------------------------------------------------------


In [50]:
response = generate_llama_response_system_prompt(
        user_prompt="What is the protocol for managing sepsis in a critical care unit?",
        system_prompt=system_prompt,
        temperature=0,
        top_p=0.9,
        top_k=40
    )

print(f"Answer:\n{response}\n")
print("-"*60)

Llama.generate: 86 prefix-match hit, remaining 19 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =      73.07 ms /    19 tokens (    3.85 ms per token,   260.02 tokens per second)
llama_perf_context_print:        eval time =   37914.28 ms /   679 runs   (   55.84 ms per token,    17.91 tokens per second)
llama_perf_context_print:       total time =   38780.18 ms /   698 tokens
llama_perf_context_print:    graphs reused =        675


Answer:
The protocol for managing sepsis in a critical care unit involves the following steps, based on current medical guidelines:

1. Early recognition and suspicion of sepsis: Identify patients who may have sepsis by assessing their vital signs, laboratory values, and clinical symptoms. Suspect sepsis if there is evidence of infection, along with signs of organ dysfunction such as altered mental status, respiratory distress, or low urine output.

2. Rapid administration of antibiotics: Once sepsis is suspected, start broad-spectrum intravenous antibiotics as soon as possible to cover potential pathogens. Adjust the antibiotic regimen based on culture and sensitivity results.

3. Fluid resuscitation: Aim for adequate tissue perfusion by administering crystalloid fluids in boluses of 20-50 mL/kg until mean arterial pressure (MAP) is >65 mmHg or central venous pressure (CVP) is between 8 and 12 cm H2O. Monitor fluid response closely to avoid overloading the patient.

4. Vasopressor sup

## Question Answering using RAG

In [53]:
user_input = "Who are the Editorial and Production Staff ?"
print(generate_rag_response_proper(user_input))

Llama.generate: 11 prefix-match hit, remaining 2101 prompt tokens to eval
llama_perf_context_print:        load time =     217.95 ms
llama_perf_context_print: prompt eval time =    3009.75 ms /  2101 tokens (    1.43 ms per token,   698.07 tokens per second)
llama_perf_context_print:        eval time =    8998.85 ms /   143 runs   (   62.93 ms per token,    15.89 tokens per second)
llama_perf_context_print:       total time =   12124.18 ms /  2244 tokens
llama_perf_context_print:    graphs reused =        142


1. Executive Editor: Keryn A.G. Lane
2. Senior Staff Writers: Susan T. Schindler, Susan C. Short
3. Staff Editor: Michelle A. Steigerwald
4. Senior Operations Manager: Diane C. Zenker
5. Senior Project Manager: Diane Cosner Gartenmayer
6. Manager, Electronic Publications: Michael A. DeFerrari
7. Executive Assistant: Jean Perry
8. Designer: Alisha Webber
9. Illustrators: Christopher C. Butts, Michael Reingold
10. Indexers: Keryn A.G. Lane, Susan Thomas, PhD


**Q1. What is the protocol for managing sepsis in a critical care unit?**

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
#print(generate_rag_response(user_input))
print(generate_rag_response_proper(user_input,max_tokens=1500))

**Q2. What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [ ]:
user_input = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response_proper(user_input,max_tokens=1500))



**Q3. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response_proper(user_input,max_tokens=1500))

**Q4. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response_proper(user_input,max_tokens=1500))

**Q5. What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response_proper(user_input,max_tokens=1500))

### Comparison of LLM Responses: Without RAG vs With RAG

The response generated **without RAG** appears fluent and confident but suffers from **hallucination**. The model provides specific author names despite having no access to the actual source document. This indicates that the answer is likely fabricated based on prior training patterns rather than grounded in factual context.

In contrast, the response generated **with RAG** is grounded in the retrieved document context. The model is able to extract and present accurate details about the editorial and production staff, including specific names and roles. This demonstrates that RAG significantly improves factual accuracy by anchoring the response to external knowledge sources.

Additionally, the RAG-based response utilizes a larger portion of the context window (as seen from higher prompt token usage), enabling it to incorporate relevant information from multiple document chunks. This results in a more reliable and context-aware answer.

 **Conclusion**

* **Without RAG:** Faster but prone to hallucinations and unreliable outputs
* **With RAG:** Slightly slower but significantly more accurate and trustworthy

Overall, RAG reduces hallucination and enhances the credibility of responses, which is especially critical for domains requiring high factual precision such as healthcare.


## Retriever and LLM Parameter tunning

In [ ]:
# RAG Parameter Tuning - 5 Experiments
# Varying k (retriever) and temperature, top_p, top_k (LLM)


def run_rag_experiments(test_question, max_tokens=1024):

    rag_experiments = [
        {"k": 3, "temperature": 0.1, "top_p": 0.95, "top_k": 50, "label": "Experiment 1 - Default"},
        {"k": 5, "temperature": 0.1, "top_p": 0.95, "top_k": 50, "label": "Experiment 2 - Higher k"},
        {"k": 3, "temperature": 0.3, "top_p": 0.95, "top_k": 50, "label": "Experiment 3 - Higher Temp"},
        {"k": 5, "temperature": 0.3, "top_p": 0.90, "top_k": 40, "label": "Experiment 4 - Higher k + Higher Temp"},
        {"k": 3, "temperature": 0.1, "top_p": 0.80, "top_k": 20, "label": "Experiment 5 - Constrained"},
    ]

    for exp in rag_experiments:
        print(f"===== {exp['label']} =====")
        print(f"k={exp['k']}, temperature={exp['temperature']}, top_p={exp['top_p']}, top_k={exp['top_k']}\n")
        response = generate_rag_response_proper(
            test_question,
            k=exp['k'],
            max_tokens=max_tokens,
            temperature=exp['temperature'],
            top_p=exp['top_p'],
            top_k=exp['top_k']
        )
        print(response)
        print("-" * 60)

**Q1. What is the protocol for managing sepsis in a critical care unit?**

In [ ]:


run_rag_experiments("What is the protocol for managing sepsis in a critical care unit?")



**Q2. What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [ ]:

run_rag_experiments("What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?")

**Q3. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [ ]:

run_rag_experiments("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?")

## RAG Parameter Tuning - Combined Summary (Q1, Q2, Q3)
| Experiment | k | Temp | Top_p | Top_k | Overall Quality |
|---|---|---|---|---|---|
| Exp 1 - Default | 3 | 0.1 | 0.95 | 50 | Good, concise |
| Exp 2 - Higher k | 5 | 0.1 | 0.95 | 50 | Best precision and completeness |
| Exp 3 - Higher Temp | 3 | 0.3 | 0.95 | 50 | Slightly better than Exp 1 |
| Exp 4 - Higher k + Temp | 5 | 0.3 | 0.90 | 40 | Most elaborate, occasionally verbose |
| Exp 5 - Constrained | 3 | 0.1 | 0.80 | 20 | Weakest, mirrors Exp 1 |

- **`k` had the highest impact** — increasing from 3 to 5 consistently added clinically
  important details missing in k=3 responses across all 3 questions.
- **Temperature had moderate impact** — mostly noticeable at k=3 for Q1 and Q2, but
  negligible for Q3 where the same response was produced across Exp 1, 3, and 5.
- **Constrained parameters added no value** — restricting top_p and top_k produced
  responses identical to the default when temperature was already low.
- **Groundedness was consistent across all experiments** — no hallucinations observed
  regardless of parameter settings, confirming RAG's reliability.

**Best configuration: k=5, temperature=0.1** — complete, precise, and grounded.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

### Defining the Evaluation Prompts

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

### Defining the Evaluation Function

In [ ]:
def generate_ground_relevance_response(user_input, k=3, max_tokens=1024, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template

    relevant_document_chunks = vectorstore.similarity_search(user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Get the RAG answer first
    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = f"""[INST] <<SYS>>
{qna_system_message}
<</SYS>>

{user_message} [/INST]"""

    response = mistral_llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=['[INST]'],
    )
    answer = response['choices'][0]['text']

    # Groundedness evaluation
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
        {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
        [/INST]"""

    # Relevance evaluation
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
        {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
        [/INST]"""

    response_1 = mistral_llm(
        prompt=groundedness_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=['[INST]'],
    )

    response_2 = mistral_llm(
        prompt=relevance_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=['[INST]'],
    )

    return response_1['choices'][0]['text'], response_2['choices'][0]['text']

**Q1. What is the protocol for managing sepsis in a critical care unit?**

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

In [ ]:
print(ground,end="\n\n")
print(rel)

**Q2. What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [ ]:
user_input = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

In [ ]:
print(ground,end="\n\n")
print(rel)

**Q3. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

In [ ]:
print(ground,end="\n\n")
print(rel)

**Q4. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [ ]:
user_input = " What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

In [ ]:
print(ground,end="\n\n")
print(rel)

**Q5. What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

In [ ]:
user_input = " What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

In [ ]:
print(ground,end="\n\n")
print(rel)

### Evaluation Summary

| Question | Groundedness | Relevance |
|---|---|---|
| Q1: Sepsis Protocol | 5/5 | 4/5 |
| Q2: Appendicitis | 5/5 | 4/5 |
| Q3: Hair Loss | 5/5 | 3/5 |
| Q4: Brain Injury | 5/5 | 5/5 |
| Q5: Leg Fracture | 5/5 | 3/5 |

**Groundedness** scored a perfect 5/5 across all questions — every answer was derived
strictly from the retrieved Merck Manual context with no hallucination.

**Relevance** was slightly lower for Q3 (Hair Loss) and Q5 (Leg Fracture), scoring 3/5.
This suggests the retrieved chunks covered the topic but missed some specific aspects of
the question — for example, Q5 lacked explicit information about precautions during
transportation. Increasing `k` from 3 to 5 for these queries could help retrieve more
complete context.

Overall, the RAG pipeline performs strongly on groundedness, confirming that the model
stays within the verified medical knowledge. Relevance scores indicate room to improve
retrieval quality for certain query types.

## Actionable Insights and Recommendations

### Observations and Insights

1. **LLM alone is unreliable for medical use** — Without RAG, the model generated
   fluent but unverifiable responses, fabricating specific values like drug dosages
   and clinical thresholds with confidence. This is dangerous in a healthcare setting.

2. **Prompt engineering improves structure but not accuracy** — A well-designed system
   prompt improved response organization and clinical tone, but the core problem of
   hallucination remained. The model was still drawing from training memory, not
   verified sources.

3. **RAG solves the hallucination problem** — Every RAG response scored 5/5 on
   groundedness, confirming that grounding the model in the Merck Manual effectively
   eliminated fabricated information across all 5 medical questions.

4. **Retrieval quality is the weakest link** — Relevance scores of 3/5 for hair loss
   and leg fracture questions indicate that the retrieved chunks didn't fully cover
   all aspects of the query. The quality of the answer is only as good as what
   gets retrieved.

5. **Low temperature is critical for medical QA** — Parameter tuning showed that
   temperature=0.1 consistently produced the most focused and clinically accurate
   responses. Higher temperatures introduced verbosity and off-topic content.

### Business Recommendations

1. **Position as a decision-support tool, not a replacement** — The system should
   assist healthcare professionals in quickly accessing relevant medical knowledge,
   not replace clinical judgment. Always include a human review layer for critical
   decisions.

2. **Expand the knowledge base** — Integrate additional verified sources such as
   WHO guidelines, CDC protocols, and recent clinical research papers alongside the
   Merck Manual to improve coverage and keep the system up to date.

3. **Improve retrieval for complex queries** — Increasing `k` from 3 to 5 for
   multi-part questions and experimenting with hybrid search (semantic + keyword)
   would improve relevance scores and response completeness.

4. **Validate with medical professionals** — LLM-as-a-judge evaluation is a useful
   proxy metric, but final validation must involve qualified clinicians to ensure
   clinical safety before any real-world deployment.

5. **Plan for scalable infrastructure** — The T4 GPU is sufficient for prototyping.
   Production deployment serving multiple users simultaneously would require
   higher-end hardware (A100/H100) or a cloud-based LLM API to meet latency
   requirements.